# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to use the `mlcroissant` library to load and explore the FAIR<sup>2</sup> dataset: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**.

### Dataset Source
This dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Install mlcroissant if it's not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant-defined dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata from the Dataset object
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
List available record sets (`@id`), their labels, and show which fields are defined under each. We use Croissant `@id`s to reference all entities.

In [ ]:
# List Record Sets and their Fields by Croissant @id

record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rset in record_sets:
    print(f"Record set @id: {rset.id}")
    if hasattr(rset, 'label'):
        print(f"  Label: {rset.label}")
    print(f"  Fields:")
    for fld in rset.fields:
        print(f"    Field @id: {fld.id}")
        if hasattr(fld, 'label'):
            print(f"      Label: {fld.label}")
    print("")

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. Use the record set and field `@id` from the overview above, referencing everything by `@id`.

In [ ]:
# List all record set @ids available (and choose the primary table)
rs_ids = [rset.id for rset in dataset.record_sets]
print("Record sets @id:", rs_ids)

# For this dataset, there is typically one main data table; you may need to adapt record_set_id below.
record_set_id = rs_ids[0] if rs_ids else None

if record_set_id is None:
    raise RuntimeError("No record sets found in the dataset.")

# Extract all records for this record set using mlcroissant
records = list(dataset.records(record_set=record_set_id))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from record set '{record_set_id}'.")
print(f"Columns (@id): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply some common steps:
- Filter to include only proteins with more than a threshold value in a numeric column
- Normalize a numeric field
- Group by a category (if exists)

Ensure you use field `@id`s for column references.

In [ ]:
# Possible numeric fields (check column names):
print(df.dtypes)

# For this dataset, let's assume protein abundance is a numeric field,
# replace '<numeric_field_id>' with the actual @id of a numeric field, e.g. 'abundance' or similar.
# You may need to look at df.columns output from above.

# Example realistic fields - adapt as appropriate:
numeric_field = None
possible_numeric_fields = [c for c in df.columns if df[c].dtype in [int, float]]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # Try converting columns that look numeric
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            if df[c].dtype in [int, float]:
                numeric_field = c
                break
        except Exception:
            continue
if numeric_field is None:
    raise RuntimeError('No numeric field found; please use the correct @id from columns!')

print(f"Chosen numeric field for analysis: {numeric_field}")

# Filter out records with numeric_field > threshold
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} (first 5 values):")
print(filtered_df[[numeric_field, normalized_col]].head())

# Try grouping by a likely category field (string/object dtype)
candidate_categories = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
group_field = candidate_categories[0] if candidate_categories else None
if group_field:
    print(f"Grouping by {group_field}")
    grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
    print("Grouped summary (first 5):")
    print(grouped_df.head())

## 5. Visualization
Plot the distribution of the selected numeric field, and the grouped means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If category grouping, plot mean by group
if group_field:
    plt.figure(figsize=(10,4))
    order = grouped_df.sort_values(numeric_field, ascending=False)[group_field]
    sns.barplot(x=group_field, y=numeric_field, data=grouped_df, order=order)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded the FAIR² proteomics dataset using its Croissant schema and the `mlcroissant` library.
- We explored available record sets, extracted tabular data, filtered and normalized protein data, and performed grouping and visualizations.
- For deeper analysis, use the printed record set and field `@id`s to precisely control data selection.

**Note:** Always reference dataset fields by their `@id` (not just label or column name) for fully portable Croissant analysis.